## Exercícios

> Retirados de [learn-python: sqlalchemy_orm-questions](https://aviadr1.github.io/learn-advanced-python/11_db_access/exercise/sqlalchemy_orm-questions.html).

#### Q1.

Baixa e extraia o arquivo compactado com o banco de dados [Chinook database](https://www.sqlitetutorial.net/sqlite-sample-database/). Salve o arquivo `chinook.db` na mesma pasta deste script.
* Link para baixar: http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip

<img width=500 src=https://www.sqlitetutorial.net/wp-content/uploads/2015/11/sqlite-sample-database-color.jpg>


#### Q2.

Leia o código e os comentários das células a seguir para entender como acessamos os modelos ORM de um banco já existente.

In [ ]:
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.orm import Session

engine = create_engine("sqlite+pysqlite:///chinook.db", echo=False)

### extrai as classes da base de dados Chinook
metadata = MetaData()
metadata.reflect(engine)

# O metadata tem informações sobre as tabelas
# que serão usadas para criar os modelos ORM
for table_name, table in metadata.tables.items():
    print(table_name)
    print(table.columns.keys())
    print(table.columns.items())
    print('-'*25)

### configura o objeto Base mapeando os modelos ORM das tabelas
from sqlalchemy.ext.automap import automap_base
Base = automap_base(metadata=metadata)
Base.prepare()

# o objeto Base tem os modelos ORM que podemos usar
# para manipular o banco de dados
print(Base.classes.items())

In [ ]:
# A seguir um exemplo de query na tabela Albums
# usamos o objeto Base para acessar cada modelo ORM.

session = Session(engine)
res = session.scalars(select(Base.classes.albums))
first_album = res.first()
print(first_album.AlbumId, first_album.Title)

#### Q3. 
Com base nos códigos anteriores realize as operações solicitadas nas células a seguir:


In [ ]:
# Imprima os tr?s primeiros registros da tabela tracks
from sqlalchemy import select

session = Session(engine)
res = session.scalars(select(Base.classes.tracks).limit(3))
for track in res:
    print(track.TrackId, track.Name, track.AlbumId, track.MediaTypeId, track.GenreId, track.Composer, track.Milliseconds, track.Bytes, track.UnitPrice)


In [ ]:
# Imprima o nome da faixa e o t?tulo do ?lbum das primeiras 20 faixas na tabela tracks.
from sqlalchemy import select

Tracks = Base.classes.tracks
Albums = Base.classes.albums

stmt = (
    select(Tracks.Name, Albums.Title)
    .join(Albums, Tracks.AlbumId == Albums.AlbumId)
    .limit(20)
)

session = Session(engine)
for name, title in session.execute(stmt):
    print(name, "-", title)


In [ ]:
# Imprima as 10 primeiras vendas de faixas da tabela invoice_items
# Para essas 10 primeiras vendas, imprima os nomes das faixas vendidas e a quantidade vendida.
from sqlalchemy import select

InvoiceItems = Base.classes.invoice_items
Tracks = Base.classes.tracks

stmt = (
    select(InvoiceItems.InvoiceLineId, Tracks.Name, InvoiceItems.Quantity)
    .join(Tracks, InvoiceItems.TrackId == Tracks.TrackId)
    .limit(10)
)

session = Session(engine)
for line_id, name, qty in session.execute(stmt):
    print(line_id, name, qty)


In [ ]:
# Imprima os nomes das 10 faixas mais vendidas e quantas vezes foram vendidas.
from sqlalchemy import select, func

InvoiceItems = Base.classes.invoice_items
Tracks = Base.classes.tracks

stmt = (
    select(Tracks.Name, func.sum(InvoiceItems.Quantity).label("total_vendas"))
    .join(Tracks, InvoiceItems.TrackId == Tracks.TrackId)
    .group_by(Tracks.TrackId)
    .order_by(func.sum(InvoiceItems.Quantity).desc())
    .limit(10)
)

session = Session(engine)
for name, total in session.execute(stmt):
    print(name, total)


In [ ]:
# Quem s?o os 10 artistas que mais venderam?
from sqlalchemy import select, func

InvoiceItems = Base.classes.invoice_items
Tracks = Base.classes.tracks
Albums = Base.classes.albums
Artists = Base.classes.artists

stmt = (
    select(Artists.Name, func.sum(InvoiceItems.Quantity).label("total_vendas"))
    .join(Albums, Artists.ArtistId == Albums.ArtistId)
    .join(Tracks, Tracks.AlbumId == Albums.AlbumId)
    .join(InvoiceItems, InvoiceItems.TrackId == Tracks.TrackId)
    .group_by(Artists.ArtistId)
    .order_by(func.sum(InvoiceItems.Quantity).desc())
    .limit(10)
)

session = Session(engine)
for name, total in session.execute(stmt):
    print(name, total)
